In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score


## 1.Chargement de la data

In [ ]:
df=pd.read_csv('../data/interim/rfmdata_cleaned.csv')

## 2. Date de référence

La date de référence est fixée au lendemain de la dernière transaction.
Elle permet de calculer la récence de chaque client.

In [ ]:
df["InvoiceDate"]=pd.to_datetime(df["InvoiceDate"])

In [ ]:
df["TotalAmount"] = df["Quantity"] * df["Price"]

In [ ]:
date_reference = df["InvoiceDate"].max() + pd.Timedelta(days=1)

## 3. Calcul du RFM

Nous regroupons les transactions par client pour calculer les trois indicateurs :

- **Recency** : temps depuis le dernier achat.
- **Frequency** : nombre de factures distinctes.
- **Monetary** : montant total dépensé.

In [ ]:
rfm = df.groupby("CustomerID").agg(
    Recency=("InvoiceDate", lambda x: (date_reference - x.max()).days),
    Frequency=("Invoice", "nunique"),
    Monetary=("TotalAmount", "sum")
)

## 4. Vérification du RFM

Nous affichons quelques lignes afin de vérifier que les trois variables RFM ont été correctement calculées.

In [ ]:
rfm.head()

## 5. Vérification des valeurs manquantes

Nous vérifions qu'il n'y a aucune valeur manquante dans les variables RFM.

In [ ]:
rfm.isnull().sum()

## 6. Statistiques descriptives

Nous observons les principales statistiques des variables RFM afin de vérifier leurs distributions et leurs valeurs.

In [ ]:
rfm.describe()

## 7. Vérification de la cohérence des valeurs

Nous vérifions que les valeurs de Recency, Frequency et Monetary sont positives et cohérentes.

In [ ]:
print("Recency minimum :", rfm["Recency"].min())
print("Frequency minimum :", rfm["Frequency"].min())
print("Monetary minimum :", rfm["Monetary"].min())

## 8. Nettoyage des valeurs RFM

Nous conservons uniquement les clients ayant des valeurs RFM valides.

In [ ]:
rfm = rfm[(rfm["Recency"] >= 0) &
          (rfm["Frequency"] > 0) &
          (rfm["Monetary"] > 0)]

## 9. Transformation logarithmique

Les variables RFM étant fortement asymétriques, nous appliquons une transformation logarithmique pour réduire l'effet des valeurs extrêmes.

In [ ]:
rfm_log = np.log1p(rfm)

## 10. Standardisation

Les variables transformées sont standardisées afin de les mettre sur une échelle comparable avant le clustering.

In [ ]:
rfm_scaled = StandardScaler().fit_transform(rfm_log)

In [ ]:
rfm.head()